In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ансамблевые модели
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor



In [ ]:
df = pd.read_csv('housing.csv')
df = df.drop('ocean_proximity', axis=1)
df = df.drop('total_bedrooms', axis=1)

print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# Проверяем пропуски после удаления
print("\nПропуски:")
print(df.isnull().sum())

# Целевая переменная
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

print(f"\nМатрица признаков X: {X.shape}")
print(f"Признаки: {list(X.columns)}")
print(f"Диапазон целевой переменной: [{y.min():.2f}, {y.max():.2f}]")

# Масштабирование признаков (для KNN и SVR, но ансамбли обычно не требуют)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"\nОбучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")

In [ ]:
# Базовый классификатор - дерево решений
base_tree = DecisionTreeRegressor(max_depth=10, random_state=42)

# Bagging регрессор
bagging = BaggingRegressor(
    estimator=base_tree,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

print("Обучение BaggingRegressor...")
bagging.fit(X_train, y_train)
y_pred_bagging = bagging.predict(X_test)

# Оценка качества
mae_bagging = mean_absolute_error(y_test, y_pred_bagging)
mse_bagging = mean_squared_error(y_test, y_pred_bagging)
rmse_bagging = np.sqrt(mse_bagging)
r2_bagging = r2_score(y_test, y_pred_bagging)

print("\n" + "="*50)
print("Bagging Regressor")
print("="*50)
print(f"Средняя абсолютная ошибка (MAE):      {mae_bagging:.2f} $")
print(f"Средняя квадратичная ошибка (MSE):    {mse_bagging:.2f}")
print(f"RMSE:                                 {rmse_bagging:.2f} $")
print(f"Коэффициент детерминации (R²):        {r2_bagging:.4f}")
print(f"Количество деревьев: {bagging.n_estimators}")

In [ ]:
# Случайный лес
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

print("\nОбучение RandomForestRegressor...")
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Оценка качества
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("\n" + "="*50)
print("Random Forest Regressor")
print("="*50)
print(f"Средняя абсолютная ошибка (MAE):      {mae_rf:.2f} $")
print(f"Средняя квадратичная ошибка (MSE):    {mse_rf:.2f}")
print(f"RMSE:                                 {rmse_rf:.2f} $")
print(f"Коэффициент детерминации (R²):        {r2_rf:.4f}")
print(f"Количество деревьев: {rf.n_estimators}")

# Важность признаков для Random Forest
feature_importance_rf = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (Random Forest):")
for idx, row in feature_importance_rf.iterrows():
    print(f"  {row['feature']:20s}: {row['importance']:.4f} ({row['importance']*100:.2f}%)")

In [ ]:
# Extra Trees (сверхслучайные деревья)
extra_trees = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

print("\nОбучение ExtraTreesRegressor...")
extra_trees.fit(X_train, y_train)
y_pred_et = extra_trees.predict(X_test)

# Оценка качества
mae_et = mean_absolute_error(y_test, y_pred_et)
mse_et = mean_squared_error(y_test, y_pred_et)
rmse_et = np.sqrt(mse_et)
r2_et = r2_score(y_test, y_pred_et)

print("\n" + "="*50)
print("Extra Trees Regressor")
print("="*50)
print(f"Средняя абсолютная ошибка (MAE):      {mae_et:.2f} $")
print(f"Средняя квадратичная ошибка (MSE):    {mse_et:.2f}")
print(f"RMSE:                                 {rmse_et:.2f} $")
print(f"Коэффициент детерминации (R²):        {r2_et:.4f}")

In [ ]:
# AdaBoost на базе дерева решений
base_tree_shallow = DecisionTreeRegressor(max_depth=3, random_state=42)

adaboost = AdaBoostRegressor(
    estimator=base_tree_shallow,
    n_estimators=100,
    learning_rate=1.0,
    loss='linear',
    random_state=42
)

print("\nОбучение AdaBoostRegressor...")
adaboost.fit(X_train, y_train)
y_pred_ada = adaboost.predict(X_test)

# Оценка качества
mae_ada = mean_absolute_error(y_test, y_pred_ada)
mse_ada = mean_squared_error(y_test, y_pred_ada)
rmse_ada = np.sqrt(mse_ada)
r2_ada = r2_score(y_test, y_pred_ada)

print("\n" + "="*50)
print("AdaBoost Regressor")
print("="*50)
print(f"Средняя абсолютная ошибка (MAE):      {mae_ada:.2f} $")
print(f"Средняя квадратичная ошибка (MSE):    {mse_ada:.2f}")
print(f"RMSE:                                 {rmse_ada:.2f} $")
print(f"Коэффициент детерминации (R²):        {r2_ada:.4f}")
print(f"Количество слабых моделей: {adaboost.n_estimators}")

# Веса оценщиков
print(f"\nВес первого оценщика: {adaboost.estimator_weights_[0]:.4f}")
print(f"Вес последнего оценщика: {adaboost.estimator_weights_[-1]:.4f}")

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42
)

print("\nОбучение GradientBoostingRegressor...")
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# Оценка качества
mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print("\n" + "="*50)
print("Gradient Boosting Regressor")
print("="*50)
print(f"Средняя абсолютная ошибка (MAE):      {mae_gb:.2f} $")
print(f"Средняя квадратичная ошибка (MSE):    {mse_gb:.2f}")
print(f"RMSE:                                 {rmse_gb:.2f} $")
print(f"Коэффициент детерминации (R²):        {r2_gb:.4f}")
print(f"Количество деревьев: {gb.n_estimators}")
print(f"Глубина деревьев: {gb.max_depth}")

# Важность признаков для Gradient Boosting
feature_importance_gb = pd.DataFrame({
    'feature': X.columns,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (Gradient Boosting):")
for idx, row in feature_importance_gb.iterrows():
    print(f"  {row['feature']:20s}: {row['importance']:.4f} ({row['importance']*100:.2f}%)")

In [ ]:
# Сбор результатов
results = pd.DataFrame({
    'Модель': [
        'Bagging (бэггинг)',
        'Random Forest (случайный лес)',
        'Extra Trees (сверхслучайные)',
        'AdaBoost',
        'Gradient Boosting'
    ],
    'MAE': [mae_bagging, mae_rf, mae_et, mae_ada, mae_gb],
    'RMSE': [rmse_bagging, rmse_rf, rmse_et, rmse_ada, rmse_gb],
    'R²': [r2_bagging, r2_rf, r2_et, r2_ada, r2_gb]
})

print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)
print(results.round({'MAE': 2, 'RMSE': 2, 'R²': 4}).to_string(index=False))

# Определение лучших моделей
best_mae_idx = results['MAE'].idxmin()
best_rmse_idx = results['RMSE'].idxmin()
best_r2_idx = results['R²'].idxmax()

print(f"\nЛучшая модель по MAE:  {results.loc[best_mae_idx, 'Модель']} ({results.loc[best_mae_idx, 'MAE']:.2f} $)")
print(f"Лучшая модель по RMSE: {results.loc[best_rmse_idx, 'Модель']} ({results.loc[best_rmse_idx, 'RMSE']:.2f} $)")
print(f"Лучшая модель по R²:   {results.loc[best_r2_idx, 'Модель']} ({results.loc[best_r2_idx, 'R²']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# График MAE
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
axes[0].bar(results['Модель'], results['MAE'], color=colors)
axes[0].set_ylabel('MAE (доллары)', fontsize=12)
axes[0].set_title('Сравнение MAE (чем меньше, тем лучше)', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

for i, (model, mae) in enumerate(zip(results['Модель'], results['MAE'])):
    axes[0].text(i, mae + 500, f'{mae:.0f}', ha='center', fontsize=9)

# График RMSE
axes[1].bar(results['Модель'], results['RMSE'], color=colors)
axes[1].set_ylabel('RMSE (доллары)', fontsize=12)
axes[1].set_title('Сравнение RMSE (чем меньше, тем лучше)', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

for i, (model, rmse) in enumerate(zip(results['Модель'], results['RMSE'])):
    axes[1].text(i, rmse + 500, f'{rmse:.0f}', ha='center', fontsize=9)

# График R²
axes[2].bar(results['Модель'], results['R²'], color=colors)
axes[2].set_ylabel('R² (коэффициент детерминации)', fontsize=12)
axes[2].set_title('Сравнение R² (чем ближе к 1, тем лучше)', fontsize=14)
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

for i, (model, r2) in enumerate(zip(results['Модель'], results['R²'])):
    axes[2].text(i, r2 + 0.02, f'{r2:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация предсказаний лучшей модели
best_model_idx = best_r2_idx
best_model_name = results.loc[best_model_idx, 'Модель']

# Получаем предсказания лучшей модели
if best_model_name == 'Random Forest (случайный лес)':
    best_predictions = y_pred_rf
    best_model = rf
elif best_model_name == 'Extra Trees (сверхслучайные)':
    best_predictions = y_pred_et
    best_model = extra_trees
elif best_model_name == 'Gradient Boosting':
    best_predictions = y_pred_gb
    best_model = gb
elif best_model_name == 'AdaBoost':
    best_predictions = y_pred_ada
    best_model = adaboost
else:
    best_predictions = y_pred_bagging
    best_model = bagging

plt.figure(figsize=(14, 5))

# График "реальные vs предсказанные"
plt.subplot(1, 2, 1)
plt.scatter(y_test, best_predictions, alpha=0.5, c=colors[best_model_idx], edgecolors='k', s=30)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения (доллары)', fontsize=12)
plt.ylabel('Предсказанные значения (доллары)', fontsize=12)
plt.title(f'{best_model_name}\nR² = {results.loc[best_model_idx, "R²"]:.4f}', fontsize=14)
plt.grid(True, alpha=0.3)

# Распределение ошибок
plt.subplot(1, 2, 2)
errors = y_test - best_predictions
plt.hist(errors, bins=50, alpha=0.7, color=colors[best_model_idx], edgecolor='black')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Ошибка предсказания (доллары)', fontsize=12)
plt.ylabel('Частота', fontsize=12)
plt.title(f'Распределение ошибок\nСредняя ошибка: {errors.mean():.2f} $', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*60)
print("ВЫВОДЫ")
print("="*60)

print("\n1. Сравнение моделей бэггинга:")
print(f"   - Bagging:      R² = {r2_bagging:.4f}, RMSE = {rmse_bagging:.2f}")
print(f"   - Random Forest: R² = {r2_rf:.4f}, RMSE = {rmse_rf:.2f}")
print(f"   - Extra Trees:   R² = {r2_et:.4f}, RMSE = {rmse_et:.2f}")
print("   Вывод: Random Forest показал лучшие результаты среди моделей бэггинга")

print("\n2. Сравнение бустинга:")
print(f"   - AdaBoost:        R² = {r2_ada:.4f}, RMSE = {rmse_ada:.2f}")
print(f"   - Gradient Boosting: R² = {r2_gb:.4f}, RMSE = {rmse_gb:.2f}")
print("   Вывод: Gradient Boosting значительно лучше AdaBoost")

print("\n3. Общее сравнение:")
print(f"   Лучшая R²: {results.loc[best_r2_idx, 'Модель']} ({results.loc[best_r2_idx, 'R²']:.4f})")
print(f"   Лучшая MAE: {results.loc[best_mae_idx, 'Модель']} ({results.loc[best_mae_idx, 'MAE']:.2f} $)")

print("\n4. Рекомендация:")
if best_r2_idx == 1 or best_r2_idx == 4:
    print("   Random Forest или Gradient Boosting - лучшие модели для данного датасета.")
    print("   Gradient Boosting особенно хорош при правильной настройке гиперпараметров.")
else:
    print(f"   {results.loc[best_r2_idx, 'Модель']} является лучшей моделью для предсказания стоимости жилья.")

In [ ]:
print(f"Лучшее K (KFold, 5 фолдов): {grid_search_kf.best_params_['n_neighbors']}")
print(f"Лучшее R² (KFold, 5 фолдов): {grid_search_kf.best_score_:.4f}\n\n")
print(f"Лучшее K (ShuffleSplit, 5 разбиений): {grid_search_ss.best_params_['n_neighbors']}")
print(f"Лучшее R² (ShuffleSplit, 5 разбиений): {grid_search_ss.best_score_:.4f}\n\n")
print(f"\nЛучшее K (RandomizedSearchCV): {random_search.best_params_['n_neighbors']}")
print(f"Лучшее R² (RandomizedSearchCV): {random_search.best_score_:.4f}")

In [ ]:
k_best = grid_search_kf.best_params_['n_neighbors']
knn_best = KNeighborsRegressor(n_neighbors=k_best)
knn_best.fit(x_train, y_train)



In [ ]:
y_pred_best = knn_best.predict(x_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
mse_best = mean_squared_error(y_test, y_pred_best)
rmse_best = np.sqrt(mse_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"Оценка оптимальной модели с K={k_best}")
print(f"средняя абсолютная ошибка: {mae_best:.4f}")
print(f"средняя квадратичная ошибка: {mse_best:.4f}")
print(f"корень из средней квадратичной ошибки: {rmse_best:.4f}")
print(f"коэффициент детерминации: {r2_best:.4f}\n\n")

print("Сравнение")
print(f"{'Метрика'} {'Исходная (K=' + str(k_initial) + ')'} {'Оптимальная (K=' + str(k_best) + ')'}")
print(f"{'средняя абсолютная ошибка'} {mae_initial:.4f} {mae_best:.4f}")
print(f"{'средняя квадратичная ошибка'} {mse_initial:.4f} {mse_best:.4f}")
print(f"{'корень из средней квадратичной ошибки'} {rmse_initial:.4f} {rmse_best:.4f}")
print(f"{'коэффициент детерминации'} {r2_initial:.4f} {r2_best:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_initial, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения Rating')
plt.ylabel('Предсказанные значения')
plt.title(f'Исходная модель (K={k_initial})\nR² = {r2_initial:.4f}')

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_best, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения Rating')
plt.ylabel('Предсказанные значения')
plt.title(f'Оптимальная модель (K={k_best})\nR² = {r2_best:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsRegressor(n_neighbors=k)
    scores = cross_val_score(knn, x_train, y_train, cv=5, scoring='r2')
    cv_scores.append(scores.mean())

plt.figure(figsize=(10, 6))
plt.plot(k_range, cv_scores, 'bo-')
plt.xlabel('Количество соседей (K)')
plt.ylabel('Средний R² (кросс-валидация)')
plt.title('Зависимость качества модели от гиперпараметра K')
plt.grid(True)
plt.show()